# Пример прогнозирования цены квартиры

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
df = pd.read_excel("data.xlsx")
base_vars = [
    'ln_total_meters',
    'rooms_count',
    'center_distance',
    'metro_time_min'
]

building_vars = [
    'relative_floor',
    'is_apartment'
]

rent_condition_vars = [
    'no_commission',
    'has_deposit',
    'utilities_included',
    'kids_allowed',
    'pets_allowed'
]

amenity_vars = [
    'has_furniture',
    'has_washing_machine',
    'has_dryer',
    'has_fridge',
    'has_tv',
    'has_conditioner',
    'has_dishwasher',
    'has_microwave',
    'has_boiler'
]

repair_vars = [
    'repair_designer',
    'repair_euro'
]

base_vars = [x for x in base_vars if x in df.columns]
building_vars = [x for x in building_vars if x in df.columns]
rent_condition_vars = [x for x in rent_condition_vars if x in df.columns]
amenity_vars = [x for x in amenity_vars if x in df.columns]
repair_vars = [x for x in repair_vars if x in df.columns]

In [ ]:
model_cols = ['price', 'ln_price'] + base_vars + building_vars + rent_condition_vars + amenity_vars + repair_vars

df_model = df[model_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
df_model['amenities_count'] = df_model[amenity_vars].sum(axis=1)

df_model.shape
formula_m6 = '''
ln_price ~ ln_total_meters
         + rooms_count
         + center_distance
         + relative_floor
         + is_apartment
         + utilities_included
         + repair_euro
         + amenities_count
'''

model_6 = smf.ols(formula_m6, data=df_model).fit()
print(model_6.summary())

                            OLS Regression Results                            
Dep. Variable:               ln_price   R-squared:                       0.523
Model:                            OLS   Adj. R-squared:                  0.519
Method:                 Least Squares   F-statistic:                     134.1
Date:                Wed, 06 May 2026   Prob (F-statistic):          1.56e-151
Time:                        19:35:17   Log-Likelihood:                -759.53
No. Observations:                 987   AIC:                             1537.
Df Residuals:                     978   BIC:                             1581.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.3224      0

## Прогноз цены для квартиры с заданными характеристиками

Для демонстрации работы модели рассмотрим такую квартиру:

- площадь: 60 м²;
- количество комнат: 2;
- расстояние до центра: 8 км;
- относительный этаж: 0.6;
- объект является апартаментами;
- коммунальные услуги включены;
- евроремонт;
- количество удобств: 6.

Для прогноза используем нашу итоговая модель 6:

$$
\ln(price_i)=
\beta_0
+\beta_1 \ln(total\_meters_i)
+\beta_2 rooms\_count_i
+\beta_3 center\_distance_i
+\beta_4 relative\_floor_i
+\beta_5 is\_apartment_i
+\beta_6 utilities\_included_i
+\beta_7 repair\_euro_i
+\beta_8 amenities\_count_i
+u_i
$$


In [ ]:
new_flat = pd.DataFrame({
    'ln_total_meters': [np.log(60)],
    'rooms_count': [2],
    'center_distance': [8],
    'relative_floor': [0.6],
    'is_apartment': [1],
    'utilities_included': [1],
    'repair_euro': [1],
    'amenities_count': [6]
})

pred_ln = model_6.predict(new_flat)[0]

pred_price = np.exp(pred_ln)

print("Predicted ln(price):", pred_ln)
print("Predicted price:", pred_price)

Predicted ln(price): 11.7450572877616
Predicted price: 126128.59841091147


модель прогнозирует, что квартира с заданными характеристиками будет сдаваться примерно за 126 тыс. руб. в месяц - результат выглядит реалистичным для уровня цен на рынке аренды жилья в Москве